# Step 4 · Check whether it works (validation 2024)

**Goal:** grade Models 1–4 the same way, on the same 26,739 campaigns from 2024, to find which is best and whether any are useful.

**Three grades, then one table (the plan's Step 4):**
1. **AUC.** Does the model rank funded campaigns above failed ones?
2. **Confusion matrix.** Once we draw a line, how many does it get right?
3. **Near-miss hit rate.** Is the call list useful to Kickstarter?

**Where we draw the line:** every model flags its **2,000 riskiest** campaigns (lowest scores). A 50% cutoff would flag a different number for each model (Model 1: 3,240, Model 4: about 4,900), so the boxes would not be comparable. `COACH_CAPACITY = 2,000` is still a placeholder for a real staffing number.

**Team rules followed:** 2024 only. The 2025 test file is not opened here. No `post_` columns.

**Before running:** put `baseline_val.parquet`, `model2_val.parquet`, `model3_val.parquet` and `model4_val.parquet` in one folder. In Colab, either mount Google Drive (next cell) or upload them to `/content/split`. Running locally works too: the notebook falls back to the current folder.

In [2]:
DRIVE_DIR = None   # not using Google Drive; the 4 files were uploaded directly to this Colab session

### PIPELINE: Load the four models' 2024 scores
Each model saved one score per 2024 campaign. The cell checks that all four files list the same campaigns in the same order. If the rows were offset, every campaign would be graded against someone else's answer and nothing would warn us.

In [3]:
import os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix

BASE = next(p for p in [DRIVE_DIR, "/content/split", "."] if p and os.path.isdir(p))

FILES = {"Model 1: lookup":           ("baseline_val", "score_model1"),
         "Model 2: logistic":         ("model2_val",   "score_model2"),
         "Model 3: boosting":         ("model3_val",   "score_model3"),
         "Model 4: boosting + blurb": ("model4_val",   "score_model4")}

scores, ref = {}, None
for name, (f, col) in FILES.items():
    d = pd.read_parquet(f"{BASE}/{f}.parquet")
    if ref is None:
        ref = d[["id", "success", "near_miss"]]
    assert (d["id"].to_numpy() == ref["id"].to_numpy()).all(), f"{f}: rows do not match"
    scores[name] = d[col].to_numpy(float)

ids     = ref["id"].to_numpy()
y       = ref["success"].to_numpy().astype(int)
nm      = ref["near_miss"].to_numpy().astype(int)
RANDOM_NM_RATE = nm.mean()
COACH_CAPACITY = 2_000          # placeholder: campaigns Kickstarter can coach in a year

print(f"reading from: {BASE}")
print(f"2024: {len(y):,} campaigns | funded {y.mean():.1%} | failed {(1 - y).sum():,} | "
      f"near-misses {nm.sum():,} ({RANDOM_NM_RATE:.2%} = the random hit rate)")

reading from: .
2024: 26,739 campaigns | funded 72.2% | failed 7,443 | near-misses 541 (2.02% = the random hit rate)


## 1. AUC: does the model rank campaigns in the right order?
Pick one funded and one failed campaign. Did the model give the funded one a higher score? AUC is the share of all such pairs it gets right: 1.0 is perfect, 0.5 is a coin flip.

The 95% range comes from resampling the 2024 campaigns 200 times and recomputing AUC each time. It shows how much the number could move from luck in which campaigns happened to launch in 2024. The same resamples give a range for **each model's gain over the one before**, which is the question that matters when gains are small.

In [4]:
rng = np.random.default_rng(0)
names = list(scores)
boot = {k: [] for k in names}
for _ in range(200):
    i = rng.integers(0, len(y), len(y))
    for k in names:
        boot[k].append(roc_auc_score(y[i], scores[k][i]))
boot = {k: np.array(v) for k, v in boot.items()}

auc = pd.DataFrame({"AUC": {k: roc_auc_score(y, scores[k]) for k in names},
                    "95% range": {k: f"{np.percentile(boot[k], 2.5):.3f}-{np.percentile(boot[k], 97.5):.3f}" for k in names}})
gain = [""] + [f"{(boot[b] - boot[a]).mean():+.4f} [{np.percentile(boot[b] - boot[a], 2.5):+.4f}, "
               f"{np.percentile(boot[b] - boot[a], 97.5):+.4f}]" for a, b in zip(names, names[1:])]
auc["gain over previous model [95% range]"] = gain
print(auc.to_string(float_format=lambda x: f"{x:.4f}"))

                             AUC    95% range gain over previous model [95% range]
Model 1: lookup           0.8067  0.801-0.812                                     
Model 2: logistic         0.8526  0.848-0.857           +0.0459 [+0.0430, +0.0486]
Model 3: boosting         0.8603  0.856-0.865           +0.0076 [+0.0059, +0.0092]
Model 4: boosting + blurb 0.8621  0.857-0.867           +0.0018 [+0.0004, +0.0034]


**Reading it.** Each model beats the one before, and every gain's range stays above zero. The gains shrink fast: Model 2 adds about +0.046 over the lookup, Models 3 and 4 add less than +0.01 each.

⚠️ **Model 3 caveat:** its settings were chosen on 2024, the year it is graded on. Its README measured that head start at +0.003 (0.857 when tuned on 2023 instead). Model 4 was tuned on 2023, so its number has no such caveat.

## 2. Confusion matrix: once we draw a line, how many does it get right?
"Flagged" = we would spend a coach's call on it, i.e. one of the model's 2,000 lowest scores. Because we are hunting failures, **failure is the "positive" outcome** here.

- **Precision:** of the 2,000 flagged, how many failed?
- **Recall:** of all 2024 failures, how many did we flag? Capped by capacity: 2,000 calls ÷ 7,443 failures = 26.9% is the most any model could reach.

In [5]:
def flag_riskiest(score, capacity=COACH_CAPACITY):
    # lowest scores first; campaign id breaks ties so the list is reproducible (Model 1 has many ties)
    flagged = np.zeros(len(score), dtype=bool)
    flagged[np.lexsort((ids, score))[:capacity]] = True
    return flagged

flags = {k: flag_riskiest(s) for k, s in scores.items()}

for k, f in flags.items():
    tn, fp, fn, tp = confusion_matrix(y == 0, f).ravel()
    box = pd.DataFrame({"flagged": [tp, fp], "not flagged": [fn, tn]},
                       index=["actually failed", "actually funded"])
    print(f"{k}\n{box.to_string()}")
    print(f"  precision {tp / (tp + fp):.1%} of flagged failed  |  recall {tp / (tp + fn):.1%} of failures caught\n")

Model 1: lookup
                 flagged  not flagged
actually failed     1632         5811
actually funded      368        18928
  precision 81.6% of flagged failed  |  recall 21.9% of failures caught

Model 2: logistic
                 flagged  not flagged
actually failed     1851         5592
actually funded      149        19147
  precision 92.5% of flagged failed  |  recall 24.9% of failures caught

Model 3: boosting
                 flagged  not flagged
actually failed     1882         5561
actually funded      118        19178
  precision 94.1% of flagged failed  |  recall 25.3% of failures caught

Model 4: boosting + blurb
                 flagged  not flagged
actually failed     1886         5557
actually funded      114        19182
  precision 94.3% of flagged failed  |  recall 25.3% of failures caught



Precision = share of the 2,000 flagged campaigns that actually failed.

Recall = share of all 7,443 failures we flagged, which can't exceed 26.9% with only 2,000 calls.

Every model beats random at finding failures (Model 4: 94.3% precision vs. 27.8% random), but most failures were never close, so this doesn't mean the list finds near-misses (see section 3).

## 3. Near-miss hit rate: is this list useful to Kickstarter?
A **near-miss** failed but raised 50–99% of its goal. These are the campaigns a coaching call might save. Of the 2,000 flagged, what share were near-misses? Compare with calling 2,000 campaigns at random, which would hit 2.02%.

In [6]:
for k, f in flags.items():
    hits = int(nm[f].sum())
    print(f"{k:<28} near-misses caught {hits:>3} of {f.sum():,}  ->  hit rate {hits / f.sum():.2%}"
          f"  ({hits / f.sum() / RANDOM_NM_RATE:.2f}x random)")
print(f"{'random 2,000 calls':<28} near-misses expected {RANDOM_NM_RATE * COACH_CAPACITY:>3.0f} of 2,000  ->  "
      f"hit rate {RANDOM_NM_RATE:.2%}  (1.00x)")

Model 1: lookup              near-misses caught  37 of 2,000  ->  hit rate 1.85%  (0.91x random)
Model 2: logistic            near-misses caught  19 of 2,000  ->  hit rate 0.95%  (0.47x random)
Model 3: boosting            near-misses caught  22 of 2,000  ->  hit rate 1.10%  (0.54x random)
Model 4: boosting + blurb    near-misses caught  24 of 2,000  ->  hit rate 1.20%  (0.59x random)
random 2,000 calls           near-misses expected  40 of 2,000  ->  hit rate 2.02%  (1.00x)


The models can tell which campaigns will fail, but not which ones almost succeed, and the near-misses are the ones Kickstarter wants to call.

An analogy. Imagine a teacher who wants to tutor students who are just below passing, scoring 60–69 on a test where 70 passes. One quick plan is to tutor the students the model thinks are most likely to fail. But that list fills up with students scoring 10 or 20. They will fail, but an hour of tutoring won’t get them to 70. The students scoring 65 aren’t on the list, because the model sees them as only maybe failing.

Same thing here:
- The riskiest campaigns do fail, but badly. A typical one might raise $300 of a $10,000 goal, and a phone call won’t fix that.
- Near-misses, like $8,500 of $10,000, get middle scores, around 40–60%. The model is unsure about them, so they never make the “riskiest” list.
- Better models are more confident about the hopeless cases, so their riskiest list holds even more of them and even fewer near-misses.

Why it matters. A model can be accurate (high AUC) and still produce the wrong call list. So in Step 5, instead of calling the riskiest campaigns, we call the ones in the middle, where near-misses actually show up.


## 4. One table (validation 2024 only)

In [7]:
rows = {}
for k in names:
    f = flags[k]
    tn, fp, fn, tp = confusion_matrix(y == 0, f).ravel()
    rows[k] = {"AUC": roc_auc_score(y, scores[k]),
               "flagged that failed (precision)": tp / (tp + fp),
               "failures caught (recall)": tp / (tp + fn),
               "near-miss hit rate": nm[f].mean(),
               "lift vs random": nm[f].mean() / RANDOM_NM_RATE}
rows["Random 2,000 calls"] = {"AUC": 0.5, "flagged that failed (precision)": 1 - y.mean(),
                              "failures caught (recall)": COACH_CAPACITY / len(y),
                              "near-miss hit rate": RANDOM_NM_RATE, "lift vs random": 1.0}
step4 = pd.DataFrame(rows).T
display_fmt = {"AUC": "{:.3f}", "flagged that failed (precision)": "{:.1%}", "failures caught (recall)": "{:.1%}",
               "near-miss hit rate": "{:.2%}", "lift vs random": "{:.2f}x"}
print(f"STEP 4: VALIDATION 2024 (used for choosing) | {COACH_CAPACITY:,} calls, riskiest first\n")
print(step4.to_string(formatters={c: f.format for c, f in display_fmt.items()}))

step4.to_csv(f"{BASE}/step4_results.csv")
print(f"\nsaved {BASE}/step4_results.csv")

STEP 4: VALIDATION 2024 (used for choosing) | 2,000 calls, riskiest first

                            AUC flagged that failed (precision) failures caught (recall) near-miss hit rate lift vs random
Model 1: lookup           0.807                           81.6%                    21.9%              1.85%          0.91x
Model 2: logistic         0.853                           92.5%                    24.9%              0.95%          0.47x
Model 3: boosting         0.860                           94.1%                    25.3%              1.10%          0.54x
Model 4: boosting + blurb 0.862                           94.3%                    25.3%              1.20%          0.59x
Random 2,000 calls        0.500                           27.8%                     7.5%              2.02%          1.00x

saved ./step4_results.csv


## Step 4 summary
- **Best model: Model 4** (AUC 0.862), with every step-up confirmed by the resampling ranges. The gains after Model 2 are real but small.
- **Useful? Not with the obvious rule.** Calling each model's 2,000 riskiest campaigns finds failures well (about 82–94% precision) but finds near-misses **less often than calling at random**.
- **Carry into Step 5:** choose the coaching rule by where near-misses cluster (the middle of the score range), set it on 2024, then test once on 2025.

# Step 5 · Decide who gets coached

**Goal:** turn Model 4's scores into an actual list of campaigns Kickstarter should call.

**Two different things, kept apart on purpose:**
- **The score** is a prediction made on launch day: "this campaign has a 47% chance of getting funded."
- **A near-miss** is what actually happened at the end: "it raised 88% of its goal and failed."

Step 4 showed that the lowest scores are *not* where near-misses live. This step finds the score range that tends to turn into near-misses, using past campaigns (2024), then tests that rule once on campaigns the model has never seen (2025).

**Order matters:**
1. Group 2024 campaigns into the plan's five score brackets.
2. Measure each bracket's hit rate: near-misses ÷ campaigns.
3. Pick the best bracket and **freeze the rule**.
4. Only then open 2025, apply the frozen rule, and compare with calling at random.

**Before running:** upload these five files to this Colab session (Files panel on the left), or put them in `/content/split`:
`model4_val.parquet`, `slice_test.parquet`, `X_test.parquet`, `blurb_scores.parquet`, `model4.joblib`.

**Team rules followed:** the rule is chosen on 2024 only. 2025 is scored once, after the rule is frozen. The `post_` columns are never model inputs.

### PIPELINE: Load Model 4's 2024 scores
These are the scores Model 4 gave every 2024 campaign, saved by the Model 4 notebook, next to what actually happened.

In [8]:
import os
import numpy as np, pandas as pd

BASE = next(p for p in ["/content/split", "."] if os.path.isdir(p) and os.path.exists(f"{p}/model4_val.parquet"))

val = pd.read_parquet(f"{BASE}/model4_val.parquet")
s_val  = val["score_model4"].to_numpy(float)
y_val  = val["success"].to_numpy(int)
nm_val = val["near_miss"].to_numpy(int)
RANDOM_2024 = nm_val.mean()

print(f"reading from: {BASE}")
print(f"2024: {len(val):,} campaigns | near-misses {nm_val.sum():,} | random hit rate {RANDOM_2024:.2%}")

reading from: .
2024: 26,739 campaigns | near-misses 541 | random hit rate 2.02%


## 1–3. Group 2024 into score brackets and check what happened
The five brackets come from the plan and were fixed **before** looking at the results. Fixing them in advance matters: if we tried many different ranges and kept the best, the winner would partly be luck.

- **campaigns:** how many 2024 campaigns Model 4 scored in that range
- **funded:** share that actually got funded (a check that the scores mean something)
- **near-misses / hit rate:** how many failed at 50–99% of goal, and that count ÷ campaigns
- **lift:** hit rate ÷ the random rate (2.02%). Above 1× beats calling at random.

In [9]:
BRACKETS = [0, .2, .4, .6, .8, 1.0001]   # 0-20%, 20-40%, 40-60%, 60-80%, 80-100% (plan Step 5)
LABELS   = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]

def bracket_table(score, success, near_miss):
    d = pd.DataFrame({"bracket": pd.cut(score, BRACKETS, right=False, labels=LABELS),
                      "success": success, "near_miss": near_miss})
    t = d.groupby("bracket", observed=False).agg(campaigns=("success", "size"),
                                                 funded=("success", "mean"),
                                                 near_misses=("near_miss", "sum"),
                                                 hit_rate=("near_miss", "mean"))
    t["lift"] = t["hit_rate"] / near_miss.mean()
    return t

fmt = {"campaigns": "{:,.0f}".format, "funded": "{:.1%}".format, "near_misses": "{:,.0f}".format,
       "hit_rate": "{:.2%}".format, "lift": "{:.2f}x".format}
b2024 = bracket_table(s_val, y_val, nm_val)
print(f"MODEL 4 SCORE BRACKETS, 2024 (random hit rate {RANDOM_2024:.2%})\n")
print(b2024.to_string(formatters=fmt))

MODEL 4 SCORE BRACKETS, 2024 (random hit rate 2.02%)

        campaigns funded near_misses hit_rate  lift
bracket                                            
0-20%       2,389   7.4%          32    1.34% 0.66x
20-40%      1,591  28.0%          44    2.77% 1.37x
40-60%      1,972  46.7%          98    4.97% 2.46x
60-80%      2,895  64.0%         108    3.73% 1.84x
80-100%    17,892  88.9%         259    1.45% 0.72x


**Reading it: near-misses cluster where the model is unsure.**
Each bracket is Model 4's launch-day score (its predicted chance of getting funded), not how much the campaign raised. Hit rate = near-misses per campaign called, so it shows what each call is worth.

- **0–20% (model expects failure):** 93% did fail, but most were never close. Only 1.3% were near-misses, below the 2.0% random rate.
- **80–100% (model expects funding):** most got funded, so only 1.4% were near-misses. This bracket has the most near-misses in total (259), but only because it holds two-thirds of all campaigns; reaching them would take nearly 18,000 calls.
- **40–60% (model unsure):** the highest hit rate, 5.0% (2.5× random). A campaign that ends up just short is one that could have gone either way, and that's where the model is unsure.

**Limitation:** this bracket reaches 98 of 2024's 541 near-misses (18%). The rule makes each call count, but reaching more near-misses would take more coaching capacity.

## 4. Pick the bracket and freeze the rule
The bracket with the highest 2024 hit rate becomes the launch-day rule. After this cell, nothing about the rule changes.

In [10]:
best = b2024["hit_rate"].idxmax()
RULE_LOW, RULE_HIGH = BRACKETS[LABELS.index(best)], min(BRACKETS[LABELS.index(best) + 1], 1.0)

print(f"RULE (frozen): on launch day, call every campaign Model 4 scores between {RULE_LOW:.0%} and {RULE_HIGH:.0%}.")
print(f"2024 evidence: {b2024.loc[best, 'campaigns']:,} campaigns, hit rate {b2024.loc[best, 'hit_rate']:.2%}, "
      f"{b2024.loc[best, 'lift']:.2f}x random")
print(f"\nlist size check: {b2024.loc[best, 'campaigns']:,} campaigns a year vs. the 2,000-call placeholder capacity")

RULE (frozen): on launch day, call every campaign Model 4 scores between 40% and 60%.
2024 evidence: 1,972 campaigns, hit rate 4.97%, 2.46x random

list size check: 1,972 campaigns a year vs. the 2,000-call placeholder capacity


## 5. The test: 2025, run once
2025 is the year nobody has looked at. The steps, in order:
1. **Score** every 2025 campaign with the saved Model 4. Its inputs are the launch-day columns (`X_test`) plus the blurb score already computed by the Model 4 notebook.
2. **Apply the frozen rule** to pick the call list.
3. **Only then read the answers** and compare the list's hit rate with calling at random.

⚠️ `model4.joblib` was saved with scikit-learn **1.9.1**. If the next cell fails to load it, run `%pip install scikit-learn==1.9.1`, restart the session (Runtime → Restart session), and run all cells again.

In [11]:
import joblib, sklearn
print(f"scikit-learn {sklearn.__version__} (model saved with 1.9.1)")

model4 = joblib.load(f"{BASE}/model4.joblib")
test   = pd.read_parquet(f"{BASE}/slice_test.parquet")
X_test = pd.read_parquet(f"{BASE}/X_test.parquet")
blurb  = (pd.read_parquet(f"{BASE}/blurb_scores.parquet")
          .query("slice == 'test'").set_index("id")["blurb_score"])

# row alignment: checked, not assumed
assert len(X_test) == len(test)
assert np.allclose(X_test["log_goal_usd"].to_numpy(float), test["log_goal_usd"].to_numpy(float)), "rows misaligned"

X4_test = X_test.assign(blurb_score=blurb.reindex(test["id"]).to_numpy())
assert X4_test["blurb_score"].notna().all(), "some 2025 campaigns have no blurb score"
assert list(X4_test.columns) == list(model4.feature_names_in_), "columns differ from what Model 4 was trained on"

s_test = model4.predict_proba(X4_test)[:, 1]      # step 1: score (no answers used yet)
call   = (s_test >= RULE_LOW) & (s_test < RULE_HIGH)   # step 2: apply the frozen rule
print(f"2025: {len(test):,} campaigns scored | {call.sum():,} land in the {best} bracket and would get a call")

scikit-learn 1.6.1 (model saved with 1.9.1)


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator _BinMapper from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator HistGradientBoostingClassifier from version 1.9.1 when using version 1.6.1. This might lead to breakin

2025: 26,887 campaigns scored | 2,030 land in the 40-60% bracket and would get a call


In [12]:
# step 3: now read what actually happened in 2025
y_test, nm_test = test["success"].to_numpy(int), test["near_miss"].to_numpy(int)
RANDOM_2025 = nm_test.mean()
hit = nm_test[call].mean()

# 95% range: resample the 2025 campaigns 1,000 times and recompute the lift each time
rng = np.random.default_rng(0)
lifts = []
for _ in range(1000):
    i = rng.integers(0, len(y_test), len(y_test))
    lifts.append(nm_test[i][call[i]].mean() / nm_test[i].mean())
lo, hi = np.percentile(lifts, [2.5, 97.5])

print(f"RESULT ON 2025 (test, run once)\n")
print(f"campaigns called          : {call.sum():,}")
print(f"near-misses among them    : {nm_test[call].sum():,}")
print(f"hit rate                  : {hit:.2%}")
print(f"random hit rate (2025)    : {RANDOM_2025:.2%}   -> {RANDOM_2025 * call.sum():.0f} near-misses expected from {call.sum():,} random calls")
print(f"lift vs random            : {hit / RANDOM_2025:.2f}x   (95% range {lo:.2f}x-{hi:.2f}x)")
print(f"share of all 2025 near-misses reached: {nm_test[call].sum() / nm_test.sum():.1%} ({nm_test[call].sum()} of {nm_test.sum()})")

RESULT ON 2025 (test, run once)

campaigns called          : 2,030
near-misses among them    : 61
hit rate                  : 3.00%
random hit rate (2025)    : 1.52%   -> 31 near-misses expected from 2,030 random calls
lift vs random            : 1.97x   (95% range 1.53x-2.45x)
share of all 2025 near-misses reached: 14.9% (61 of 410)


### EXPLORATION: Did the hump hold in 2025, and how does the obvious rule compare?
Shown for transparency only. These tables do **not** change the rule: choosing a new bracket after seeing 2025 would turn the test into another round of tuning.

In [13]:
print(f"MODEL 4 SCORE BRACKETS, 2025 (random hit rate {RANDOM_2025:.2%})\n")
print(bracket_table(s_test, y_test, nm_test).to_string(formatters=fmt))

ids = test["id"].to_numpy()
risky = np.zeros(len(s_test), bool)
risky[np.lexsort((ids, s_test))[:call.sum()]] = True     # same number of calls, riskiest first
print(f"\nsame number of calls ({call.sum():,}), two rules, 2025:")
print(f"  {best} bracket (our rule) : {nm_test[call].sum():>3} near-misses  ({nm_test[call].mean():.2%}, {nm_test[call].mean() / RANDOM_2025:.2f}x random)")
print(f"  riskiest first           : {nm_test[risky].sum():>3} near-misses  ({nm_test[risky].mean():.2%}, {nm_test[risky].mean() / RANDOM_2025:.2f}x random)")
print(f"  random calls (expected)  : {RANDOM_2025 * call.sum():>3.0f} near-misses  ({RANDOM_2025:.2%}, 1.00x)")

print(f"\ncalibration: Model 4 predicted {s_test.mean():.1%} of 2025 campaigns would be funded; {y_test.mean():.1%} were "
      f"({100 * (s_test.mean() - y_test.mean()):+.1f} points)")

MODEL 4 SCORE BRACKETS, 2025 (random hit rate 1.52%)

        campaigns funded near_misses hit_rate  lift
bracket                                            
0-20%       2,300   7.4%          23    1.00% 0.66x
20-40%      1,633  24.2%          26    1.59% 1.04x
40-60%      2,030  43.5%          61    3.00% 1.97x
60-80%      2,940  63.6%          88    2.99% 1.96x
80-100%    17,984  87.3%         212    1.18% 0.77x

same number of calls (2,030), two rules, 2025:
  40-60% bracket (our rule) :  61 near-misses  (3.00%, 1.97x random)
  riskiest first           :  18 near-misses  (0.89%, 0.58x random)
  random calls (expected)  :  31 near-misses  (1.52%, 1.00x)

calibration: Model 4 predicted 79.0% of 2025 campaigns would be funded; 70.7% were (+8.3 points)


### PIPELINE: Save the 2025 call list and the results
- `step5_brackets_2024.csv`: the table the rule was chosen from
- `step5_call_list_2025.csv`: every 2025 campaign the rule would call, with its score and what happened (for Step 6)
- `step5_test_scores_2025.parquet`: Model 4's score for every 2025 campaign

In [14]:
b2024.to_csv(f"{BASE}/step5_brackets_2024.csv")
cols = ["id", "name", "subcategory", "goal_usd", "success", "near_miss", "post_pct_funded", "post_usd_pledged"]
(test.loc[call, [c for c in cols if c in test.columns]]
     .assign(score_model4=s_test[call])
     .sort_values("score_model4")
     .to_csv(f"{BASE}/step5_call_list_2025.csv", index=False))
pd.DataFrame({"id": ids, "score_model4": s_test, "called": call}).to_parquet(f"{BASE}/step5_test_scores_2025.parquet", index=False)
print("saved step5_brackets_2024.csv, step5_call_list_2025.csv, step5_test_scores_2025.parquet")

saved step5_brackets_2024.csv, step5_call_list_2025.csv, step5_test_scores_2025.parquet


## Step 5 summary
- **The rule:** on launch day, call every campaign Model 4 scores **40–60%**. It was chosen on 2024 because that bracket had the highest near-miss rate, about 2.5× random.
- **The test (2025, run once):** the rule held. The call list found near-misses about **2× as often as calling at random**, and the 95% range stays well above 1×.
- **The obvious rule loses again:** with the same number of calls, "riskiest first" finds fewer near-misses than random.
- **Caveats:** near-misses were rarer in 2025 (1.5% vs 2.0% in 2024), so compare lifts, not raw hit rates. Model 4 runs about 8 points optimistic on 2025, so read scores as a ranking, not a true probability. Coaching capacity (2,000) is still a placeholder; this year's list is about 2,000 by coincidence.

# Step 6 · Show it matters

**Goal:** turn the 2025 test result into dollars for Kickstarter.

**The plan's formula:** extra campaigns funded × average amount raised × Kickstarter fee.

**Two comparisons, because they answer different questions:**
- **Coaching vs. no coaching:** what the coaching program is worth.
- **Model's list vs. calling at random:** what the *model* adds. This is the AI's contribution.

**Facts vs. assumptions.** Every input below is labelled. The facts come from Step 5's 2025 test. The biggest unknown is how many of the near-misses a coach would actually save, so the payoff is shown across a range instead of as one number.

**Before running:** upload these two files to this Colab session (Files panel on the left), or put them in `/content/split`:
`slice_test.parquet` (2025 campaigns and outcomes) and `step5_test_scores_2025.parquet` (saved by the Step 5 notebook).

**Team rules followed:** nothing is re-tuned here. The `post_` columns describe what happened after launch, so they are used only to price outcomes, never to choose who gets called.

### PIPELINE: Load the 2025 call list from Step 5

In [15]:
import os
import numpy as np, pandas as pd

BASE = next(p for p in ["/content/split", "."] if os.path.isdir(p) and os.path.exists(f"{p}/slice_test.parquet"))

test   = pd.read_parquet(f"{BASE}/slice_test.parquet")
scores = pd.read_parquet(f"{BASE}/step5_test_scores_2025.parquet")
assert (scores["id"].to_numpy() == test["id"].to_numpy()).all(), "rows do not match"

d = test.assign(score_model4=scores["score_model4"].to_numpy(), called=scores["called"].to_numpy())
nm = d["near_miss"] == 1
print(f"reading from: {BASE}")
print(f"2025: {len(d):,} campaigns | called by the Step 5 rule: {d['called'].sum():,} | near-misses in 2025: {nm.sum():,}")

reading from: .
2025: 26,887 campaigns | called by the Step 5 rule: 2,030 | near-misses in 2025: 410


## 1. The facts (measured on 2025)
- **Near-misses on the call list:** how many of the campaigns we would call really ended at 50–99% of goal.
- **Random comparison:** how many near-misses the same number of random calls would find on average.
- **Their goals:** if a coach pushes a near-miss over the line, Kickstarter collects its fee on at least the goal, because funding is all-or-nothing.

In [16]:
N_CALLS      = int(d["called"].sum())
RANDOM_RATE  = nm.mean()
list_nm      = d[d["called"] & nm]
goals_list   = list_nm["goal_usd"]
NM_LIST      = len(list_nm)
NM_RANDOM    = N_CALLS * RANDOM_RATE                 # expected near-misses from the same number of random calls
GOAL_RANDOM  = d.loc[nm, "goal_usd"].mean()          # a random near-miss's average goal

facts = pd.Series({
    "campaigns called (Step 5 rule)":              f"{N_CALLS:,}",
    "near-misses on the list":                     f"{NM_LIST}",
    "near-misses expected from random calls":      f"{NM_RANDOM:.0f}",
    "total goal of the list's near-misses":        f"${goals_list.sum():,.0f}",
    "average goal (list near-misses)":             f"${goals_list.mean():,.0f}",
    "median goal (list near-misses)":              f"${goals_list.median():,.0f}",
    "average goal (any 2025 near-miss)":           f"${GOAL_RANDOM:,.0f}",
    "largest single goal, share of the total":     f"${goals_list.max():,.0f} ({goals_list.max() / goals_list.sum():.0%})",
})
print(facts.to_string())

campaigns called (Step 5 rule)                      2,030
near-misses on the list                                61
near-misses expected from random calls                 31
total goal of the list's near-misses             $871,770
average goal (list near-misses)                   $14,291
median goal (list near-misses)                     $5,014
average goal (any 2025 near-miss)                  $7,653
largest single goal, share of the total    $118,392 (14%)


**Reading it:** The model's 2,030-campaign list contains 61 near-misses. Picking 2,030 campaigns at random, with no model, would find about 31, since only 1.5% of 2025 campaigns were near-misses. So the model finds twice as many.

The model's near-misses are also bigger: their average goal is $14,300, compared with $7,700 across all 2025 near-misses. But a few very large campaigns pull that average up. The middle campaign on the list asked for only about $5,000, and the largest ($118,000) accounts for 14% of the list's total. The dollar payoff therefore depends heavily on whether coaches save one of these few big campaigns.

## 2. The assumptions
| Input | Value | Type |
|---|---|---|
| Kickstarter fee | 5% of funds raised, on funded campaigns only | **fact** (Kickstarter help centre) |
| Amount raised if saved | the campaign's goal | **assumption**, a conservative floor: a saved campaign raises *at least* its goal |
| Save rate: share of called near-misses a coach pushes over the line | 10%, 20%, 30%, 50% | **assumption**, the weakest input; no data exists on it |
| Cost per call | not assumed: section 4 solves for the break-even cost instead | **team decision** |

Only near-misses are counted as savable. A call might also rescue a campaign that would have ended at 40%, or nudge one that would have been funded anyway to raise more. Leaving those out keeps the estimate on the low side.

In [17]:
KS_FEE     = 0.05
SAVE_RATES = [0.10, 0.20, 0.30, 0.50]

## 3. The payoff
For each save rate:
- **extra funded** = near-misses on the list × save rate
- **creator $ unlocked** = the goals of those saved campaigns: money creators get instead of nothing
- **Kickstarter fee** = creator $ unlocked × 5%
- **model's extra fee vs random** = the list's fee minus what the same number of random calls would earn

In [18]:
rows = []
for r in SAVE_RATES:
    fee_list   = goals_list.sum() * r * KS_FEE
    fee_random = NM_RANDOM * r * GOAL_RANDOM * KS_FEE
    rows.append({"save rate": f"{r:.0%}",
                 "extra funded (list)":         NM_LIST * r,
                 "extra funded (random)":       NM_RANDOM * r,
                 "creator $ unlocked (list)":   goals_list.sum() * r,
                 "Kickstarter fee (list)":      fee_list,
                 "Kickstarter fee (random)":    fee_random,
                 "model's extra fee vs random": fee_list - fee_random})
payoff = pd.DataFrame(rows).set_index("save rate")

money = "${:,.0f}".format
fmt = {c: (money if "$" in c or "fee" in c else "{:,.0f}".format) for c in payoff.columns}
print(f"PAYOFF ON 2025: {N_CALLS:,} calls, Kickstarter fee {KS_FEE:.0%}\n")
print(payoff.to_string(formatters=fmt))

PAYOFF ON 2025: 2,030 calls, Kickstarter fee 5%

          extra funded (list) extra funded (random) creator $ unlocked (list) Kickstarter fee (list) Kickstarter fee (random) model's extra fee vs random
save rate                                                                                                                                                
10%                         6                     3                   $87,177                 $4,359                   $1,185                      $3,174
20%                        12                     6                  $174,354                 $8,718                   $2,369                      $6,349
30%                        18                     9                  $261,531                $13,077                   $3,554                      $9,523
50%                        30                    15                  $435,885                $21,794                   $5,923                     $15,871


### EXPLORATION: How much do a few large campaigns drive this?
The same 20% calculation with the single largest campaign removed, and with every saved campaign valued at the median goal instead of its own.

In [19]:
r = 0.20
check = pd.Series({
    "as calculated (each campaign's own goal)": goals_list.sum() * r * KS_FEE,
    "without the largest campaign":            (goals_list.sum() - goals_list.max()) * r * KS_FEE,
    "every campaign at the median goal":        NM_LIST * goals_list.median() * r * KS_FEE,
})
print(f"Kickstarter fee at a {r:.0%} save rate\n")
print(check.map(money).to_string())

Kickstarter fee at a 20% save rate

as calculated (each campaign's own goal)    $8,718
without the largest campaign                $7,534
every campaign at the median goal           $3,058


## 4. Is it worth the cost of the calls?
We don't know what a call costs, so instead we solve for the **break-even cost per call**: the most Kickstarter could spend per campaign contacted and still not lose money on fees alone.

> break-even cost per call = Kickstarter fee ÷ number of calls

In [20]:
be = pd.DataFrame({
    "fee from the program": payoff["Kickstarter fee (list)"],
    "break-even $ per call (program vs no coaching)": payoff["Kickstarter fee (list)"] / N_CALLS,
    "break-even $ per call (model vs random list)":   payoff["model's extra fee vs random"] / N_CALLS,
})
print(be.to_string(formatters={c: "${:,.2f}".format for c in be.columns}))

# for scale: a 15-minute call at a few hourly staff costs
print("\nfor scale, a 15-minute call costs:")
for hourly in [30, 50, 80]:
    print(f"  ${hourly}/hour staff -> ${hourly / 4:.2f} per call -> {N_CALLS:,} calls cost ${N_CALLS * hourly / 4:,.0f}")

          fee from the program break-even $ per call (program vs no coaching) break-even $ per call (model vs random list)
save rate                                                                                                                 
10%                  $4,358.85                                          $2.15                                        $1.56
20%                  $8,717.70                                          $4.29                                        $3.13
30%                 $13,076.55                                          $6.44                                        $4.69
50%                 $21,794.25                                         $10.74                                        $7.82

for scale, a 15-minute call costs:
  $30/hour staff -> $7.50 per call -> 2,030 calls cost $15,225
  $50/hour staff -> $12.50 per call -> 2,030 calls cost $25,375
  $80/hour staff -> $20.00 per call -> 2,030 calls cost $40,600


**Reading it:** on Kickstarter's fee alone, a 15-minute staff call pays for itself only in the most optimistic case: if coaches save half the near-misses they call *and* staff cost about $30/hour or less. At 10–30% save rates, the break-even is $2–6 per call, below the cost of any staff call.

The program clearly pays only through a cheap channel, such as an automated email or an in-app tip sent to campaigns the model scores 40–60%, or if Kickstarter counts value beyond this year's fee (see the summary).

### PIPELINE: Save the results

In [21]:
payoff.join(be.drop(columns="fee from the program")).to_csv(f"{BASE}/step6_payoff.csv")
print(f"saved {BASE}/step6_payoff.csv")

saved ./step6_payoff.csv


## Step 6 summary
- **The model doubles what coaching can reach.** Its 2,030-campaign list held 61 near-misses, against about 31 from 2,030 campaigns picked at random, and those near-misses had larger goals (\$871,770 in total).
- **The fee payoff is modest.** If coaches save 20% of the near-misses they contact, Kickstarter earns about **\$8,700** in extra fees. About **\$6,300** of that comes from using the model instead of picking campaigns at random. Across 10–50% save rates, the range is \$4,400–\$21,800.
- **Creators gain far more than Kickstarter does.** At a 20% save rate, about **\$174,000** reaches creators who would otherwise get nothing (all-or-nothing funding). That gives creators a reason to come back, a value this one-year fee estimate does not count.
- **Recommendation:** use the model to target a **low-cost** intervention, such as an automated email or in-app tip for campaigns scored 40–60%. A staff phone call breaks even only if coaches save half the near-misses they call and staff cost about \$30/hour or less.
- **Caveats:**
  - The save rate is unknown, and it drives every dollar figure.
  - A few large campaigns make up much of the total. Valuing every saved campaign at the median goal cuts the 20% fee from \$8,700 to about \$3,100.
  - Saved campaigns are valued at their goal, a conservative floor.
  - Campaigns rescued from below 50% of goal are not counted.